# Лабораторная работа №2
## Очистка данных и подготовка признаков для моделирования

**Дисциплина:** Анализ данных и искусственный интеллект

**Датасет:** [Student Mental Health & Burnout (1M)](https://www.kaggle.com/datasets/ayeshasiddiqa123/student-health) — Kaggle

### Цель работы
Научиться готовить данные для алгоритмов машинного обучения: выполнить очистку, кодирование категориальных признаков, масштабирование числовых признаков, создание новых признаков и отбор релевантных.

### План работы
1. Исправление замечаний по ЛР №1.
2. Анализ категориальных признаков.
3. Кодирование категориальных признаков (сравнение ≥ 2 методов).
4. Масштабирование числовых признаков (сравнение ≥ 2 методов).
5. Создание новых признаков (≥ 2 новых).
6. Удаление нерелевантных признаков с обоснованием.
7. Финальный датасет для модели.

## 1. Импорты и настройки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, LabelEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42

%matplotlib inline

## 2. Загрузка данных и исправление замечаний по ЛР №1

В ЛР №1 были выявлены следующие недочёты, которые необходимо исправить:

| № | Замечание | Исправление в ЛР №2 |
|---|---|---|
| 1 | Признак `sleep_study_ratio` содержал огромные выбросы (max ≈ 102 179) из-за деления на значения, близкие к нулю | Использовать формулу `sleep / (study + 1)` и клиппинг |
| 2 | Z-score показывал одинаковое число выбросов для всех признаков — баг в применении маски | Применять маску по каждому столбцу отдельно |
| 3 | Логика очистки и создания признаков не была оформлена в переиспользуемые функции | Вынести все этапы в модульные функции |
| 4 | Не было разделения на числовые/категориальные признаки для дальнейшего моделирования | Явное определение списков `NUM_COLS` / `CAT_COLS` |

In [ ]:
def load_data(path: str, nrows: int | None = None) -> pd.DataFrame:
    """Загрузка датасета из CSV."""
    df = pd.read_csv(path, nrows=nrows)
    print(f'Загружено: {df.shape[0]:,} строк, {df.shape[1]} столбцов')
    return df

df_raw = load_data('../student_mental_health_burnout_1M.csv')
df_raw.head()

In [ ]:
def clip_outliers_iqr(data: pd.DataFrame, columns: list[str], k: float = 1.5) -> pd.DataFrame:
    """Ограничение (winsorization) выбросов по IQR-границам для указанных столбцов."""
    data = data.copy()
    q1 = data[columns].quantile(0.25)
    q3 = data[columns].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    data[columns] = data[columns].clip(lower=lower, upper=upper, axis=1)
    return data


def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    """Базовая очистка: удаление дубликатов и клиппинг выбросов числовых признаков."""
    df = df.drop_duplicates().reset_index(drop=True)
    exclude = {'age', 'academic_year'}
    num_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in exclude]
    df = clip_outliers_iqr(df, num_cols)
    return df

df = basic_clean(df_raw)
print(f'После очистки: {df.shape}')
df.describe().round(2)

## 3. Анализ категориальных признаков

В датасете присутствуют категориальные/дискретные признаки:

| Признак | Тип | Значения | Природа |
|---|---|---|---|
| `gender` | номинальный | Male / Female / Other | без порядка |
| `risk_level` | порядковый | Low < Medium < High | есть порядок |
| `academic_year` | порядковый | 1, 2, 3, 4 | дискретный, с порядком |

В разделе 5 будет добавлен порядковый признак `age_group`.

In [ ]:
def analyze_categorical(df: pd.DataFrame, cols: list[str]) -> None:
    """Выводит частоты и долю по категориальным признакам."""
    for col in cols:
        print(f'=== {col} ===')
        vc = df[col].value_counts(dropna=False)
        pct = (vc / len(df) * 100).round(2)
        print(pd.DataFrame({'count': vc, 'percent': pct}))
        print()

cat_cols_initial = ['gender', 'risk_level', 'academic_year']
analyze_categorical(df, cat_cols_initial)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, cat_cols_initial):
    order = None
    if col == 'risk_level':
        order = ['Low', 'Medium', 'High']
    sns.countplot(data=df, x=col, order=order, ax=ax, palette='Set2')
    ax.set_title(f'Распределение: {col}')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}',
                    (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Средний уровень выгорания и риска отчисления по категориям
agg_target = df.groupby('risk_level', observed=True)[['burnout_score', 'mental_health_index', 'dropout_risk']].mean().round(2)
agg_target = agg_target.reindex(['Low', 'Medium', 'High'])
print('Средние значения целевых признаков по risk_level:')
agg_target

**Наблюдения:**
- `gender` распределён почти равномерно между Male/Female, Other — редкий класс (~4%).
- `risk_level` сильно несбалансирован: Low доминирует (76.7%), High — всего 1.5%. При моделировании классификации потребуется учитывать дисбаланс.
- `academic_year` распределён равномерно (~25% на курс).
- `risk_level` имеет явную монотонную связь с `burnout_score` и `dropout_risk` → подходит для порядкового кодирования.

## 4. Кодирование категориальных признаков

Применим и сравним **3 метода** кодирования:

| Метод | Назначение | Применяем к |
|---|---|---|
| **One-Hot Encoding** | номинальные признаки без порядка | `gender` |
| **Ordinal Encoding** | признаки с естественным порядком | `risk_level` |
| **Label Encoding** | сравнение: наивное кодирование целым числом | `gender`, `risk_level` |

In [ ]:
def encode_onehot(df: pd.DataFrame, cols: list[str], drop: str | None = 'first') -> tuple[pd.DataFrame, OneHotEncoder]:
    """One-Hot Encoding для номинальных признаков. Возвращает датафрейм и обученный encoder."""
    encoder = OneHotEncoder(drop=drop, sparse_output=False, handle_unknown='ignore')
    encoded = encoder.fit_transform(df[cols])
    feature_names = encoder.get_feature_names_out(cols)
    encoded_df = pd.DataFrame(encoded, columns=feature_names, index=df.index)
    result = df.drop(columns=cols).join(encoded_df)
    return result, encoder


def encode_ordinal(df: pd.DataFrame, mapping: dict[str, list]) -> tuple[pd.DataFrame, OrdinalEncoder]:
    """Ordinal Encoding с явным заданием порядка категорий.

    mapping: {column_name: [ordered_categories]}
    """
    cols = list(mapping.keys())
    categories = [mapping[c] for c in cols]
    encoder = OrdinalEncoder(categories=categories, handle_unknown='use_encoded_value', unknown_value=-1)
    result = df.copy()
    result[cols] = encoder.fit_transform(result[cols])
    return result, encoder


def encode_label(df: pd.DataFrame, cols: list[str]) -> tuple[pd.DataFrame, dict[str, LabelEncoder]]:
    """Label Encoding — отдельный LabelEncoder для каждого столбца."""
    result = df.copy()
    encoders = {}
    for col in cols:
        le = LabelEncoder()
        result[col] = le.fit_transform(result[col].astype(str))
        encoders[col] = le
    return result, encoders

In [ ]:
# --- Вариант А: One-Hot для gender + Ordinal для risk_level (рекомендуемый) ---
df_ohe, ohe_encoder = encode_onehot(df, cols=['gender'], drop='first')
df_ohe, ord_encoder = encode_ordinal(df_ohe, mapping={'risk_level': ['Low', 'Medium', 'High']})

print('Вариант А (OHE + Ordinal) — добавленные/изменённые столбцы:')
cols_changed_A = [c for c in df_ohe.columns if c not in df.columns] + ['risk_level']
print(df_ohe[cols_changed_A].head())
print(f'\nРазмерность: {df_ohe.shape}')

In [ ]:
# --- Вариант B: Label Encoding для обоих признаков (для сравнения) ---
df_lbl, label_encoders = encode_label(df, cols=['gender', 'risk_level'])
print('Вариант B (Label Encoding):')
print(df_lbl[['gender', 'risk_level']].head())
print('\nСоответствие классов:')
for col, le in label_encoders.items():
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'\nРазмерность: {df_lbl.shape}')

In [ ]:
# Сравнение качества кодирования: корреляция закодированных признаков с burnout_score
comparison = pd.DataFrame({
    'OHE gender_Male ~ burnout':     [df_ohe['gender_Male'].corr(df['burnout_score'])],
    'OHE gender_Other ~ burnout':    [df_ohe['gender_Other'].corr(df['burnout_score'])],
    'Label gender ~ burnout':        [df_lbl['gender'].corr(df['burnout_score'])],
    'Ordinal risk_level ~ burnout':  [df_ohe['risk_level'].corr(df['burnout_score'])],
    'Label risk_level ~ burnout':    [df_lbl['risk_level'].corr(df['burnout_score'])],
}).T.rename(columns={0: 'corr'}).round(4)

print('Корреляция закодированного признака с burnout_score:')
comparison

**Сравнение методов:**

- **One-Hot Encoding** для `gender` — корректен (нет ложного порядка между Male / Female / Other). `drop='first'` избавляет от мультиколлинеарности.
- **Ordinal Encoding** для `risk_level` даёт корреляцию ≈ +0.82 с `burnout_score` — метод сохраняет осмысленный порядок Low → Medium → High.
- **Label Encoding** для `risk_level` случайно назначает метки по алфавиту (High=0, Low=1, Medium=2) → теряется порядок, корреляция искажается.
- **Label Encoding** для `gender` — плохо: вносит ложный порядок Female < Male < Other, который модель интерпретирует буквально.

**Выбор:** для финального датасета используем **Вариант А** (OHE для `gender`, Ordinal для `risk_level`).

## 5. Создание новых признаков

Дополнительно к созданным в ЛР №1 добавим **новые признаки** из предметной области:

| Признак | Формула | Смысл |
|---|---|---|
| `psych_load_index` | (stress + anxiety + depression) / 3 | агрегированная психологическая нагрузка |
| `sleep_study_balance` | sleep_hours / (study_hours + 1) | **исправленная версия** `sleep_study_ratio` из ЛР №1 |
| `digital_load` | screen_time + internet_usage | суммарная цифровая нагрузка |
| `external_pressure` | (exam + financial + family) / 3 | внешнее давление |
| `support_to_stress` | social_support / (stress_level + 1) | защитный ресурс против стресса *(новый)* |
| `is_sleep_deprived` | sleep_hours < 6 | бинарный признак депривации сна *(новый)* |
| `age_group` | бины по возрасту | возрастные группы 17–19, 20–22, 23–25, 26–29 |

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Создание дополнительных признаков из предметной области."""
    df = df.copy()

    df['psych_load_index'] = (
        df['stress_level'] + df['anxiety_score'] + df['depression_score']
    ) / 3

    # Исправленная версия sleep_study_ratio: без деления на 0
    df['sleep_study_balance'] = df['sleep_hours'] / (df['study_hours_per_day'] + 1)

    df['digital_load'] = df['screen_time'] + df['internet_usage']

    df['external_pressure'] = (
        df['exam_pressure'] + df['financial_stress'] + df['family_expectation']
    ) / 3

    # Новый признак: отношение поддержки к стрессу
    df['support_to_stress'] = df['social_support'] / (df['stress_level'] + 1)

    # Новый бинарный признак: депривация сна (< 6 часов)
    df['is_sleep_deprived'] = (df['sleep_hours'] < 6).astype(int)

    # Возрастные группы
    df['age_group'] = pd.cut(
        df['age'],
        bins=[16, 19, 22, 25, 30],
        labels=['17-19', '20-22', '23-25', '26-29']
    )

    return df

df_feat = add_features(df_ohe)

new_features = ['psych_load_index', 'sleep_study_balance', 'digital_load',
                'external_pressure', 'support_to_stress', 'is_sleep_deprived', 'age_group']
print('Статистика новых признаков:')
df_feat[new_features].describe(include='all').round(2)

In [ ]:
# Закодируем age_group как Ordinal (порядок групп по возрасту)
df_feat, _ = encode_ordinal(df_feat, mapping={'age_group': ['17-19', '20-22', '23-25', '26-29']})
print('age_group после Ordinal Encoding:')
df_feat['age_group'].value_counts().sort_index()

In [ ]:
# Визуализация: связь новых признаков с burnout_score
sample = df_feat.sample(n=10_000, random_state=RANDOM_STATE)

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

plots = [
    ('psych_load_index', 'burnout_score', 'scatter'),
    ('support_to_stress', 'burnout_score', 'scatter'),
    ('sleep_study_balance', 'burnout_score', 'scatter'),
    ('digital_load', 'burnout_score', 'scatter'),
    ('external_pressure', 'burnout_score', 'scatter'),
    ('is_sleep_deprived', 'burnout_score', 'box'),
]

for ax, (x, y, kind) in zip(axes.ravel(), plots):
    if kind == 'scatter':
        sns.scatterplot(data=sample, x=x, y=y, alpha=0.3, s=10, ax=ax, color='steelblue')
    else:
        sns.boxplot(data=df_feat, x=x, y=y, ax=ax, palette='Set2')
    ax.set_title(f'{x} vs {y}')

plt.tight_layout()
plt.show()

## 6. Удаление нерелевантных признаков

### Обоснование отбора

| Признак | Статус | Обоснование |
|---|---|---|
| `mental_health_index` | **удалить** | производная от `burnout_score` и `stress_level`; корреляция ≈ −0.9 с `burnout` → data leakage при моделировании |
| `dropout_risk` | **удалить** | рассчитывается из тех же факторов, что и целевая переменная → data leakage |
| `risk_level` | **удалить** | порядковая категория, полученная бинаризацией `burnout_score` → data leakage |
| остальные | **оставить** | содержательные независимые признаки |

**Целевая переменная:** `burnout_score` (задача регрессии).

Дополнительно проверим признаки на низкую дисперсию (`VarianceThreshold`), чтобы исключить константные.

In [ ]:
def drop_leaky_features(df: pd.DataFrame, target: str,
                        leaky: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    """Отделяет целевую переменную и удаляет признаки-утечки."""
    y = df[target].copy()
    X = df.drop(columns=[target] + leaky)
    return X, y


def drop_low_variance(df: pd.DataFrame, threshold: float = 0.0) -> pd.DataFrame:
    """Удаляет числовые признаки с дисперсией ниже порога."""
    selector = VarianceThreshold(threshold=threshold)
    selector.fit(df)
    keep = df.columns[selector.get_support()]
    dropped = sorted(set(df.columns) - set(keep))
    if dropped:
        print(f'Удалены признаки с дисперсией ≤ {threshold}: {dropped}')
    return df[keep]

TARGET = 'burnout_score'
LEAKY = ['mental_health_index', 'dropout_risk', 'risk_level']

X, y = drop_leaky_features(df_feat, target=TARGET, leaky=LEAKY)
X = drop_low_variance(X, threshold=0.0)

print(f'\nПризнаков в X: {X.shape[1]}')
print(f'Размер X: {X.shape}, y: {y.shape}')
print(f'\nСтолбцы X:')
print(list(X.columns))

In [ ]:
# Корреляция оставшихся признаков с целевой переменной
corr_with_target = X.corrwith(y).sort_values(key=abs, ascending=False)
print('Корреляция признаков с целевой burnout_score:')
print(corr_with_target.round(3))

## 7. Масштабирование числовых признаков

Для алгоритмов, чувствительных к масштабу (линейные модели, KNN, SVM, нейронные сети), требуется масштабирование.

Сравним **3 метода** `sklearn.preprocessing`:

| Метод | Формула | Когда использовать |
|---|---|---|
| **StandardScaler** | (x − μ) / σ | нормально распределённые признаки |
| **MinMaxScaler** | (x − min) / (max − min) | признаки с известными границами, нужен диапазон [0, 1] |
| **RobustScaler** | (x − median) / IQR | при наличии выбросов |

In [ ]:
SCALERS = {
    'standard': StandardScaler(),
    'minmax':   MinMaxScaler(),
    'robust':   RobustScaler(),
}


def scale_features(df: pd.DataFrame, cols: list[str], method: str = 'standard') -> tuple[pd.DataFrame, object]:
    """Масштабирование указанных столбцов выбранным методом."""
    if method not in SCALERS:
        raise ValueError(f'Unknown method: {method}. Available: {list(SCALERS)}')
    scaler = SCALERS[method].__class__()
    result = df.copy()
    result[cols] = scaler.fit_transform(result[cols])
    return result, scaler

# Определим столбцы для масштабирования: только непрерывные числовые
# Исключаем бинарные (is_sleep_deprived, gender_*) и уже закодированные порядковые (risk_level, age_group)
binary_cols = [c for c in X.columns if X[c].dropna().isin([0, 1]).all()]
ordinal_cols = ['age_group']  # academic_year тоже порядковый, но со значениями 1..4 — масштабировать можно
num_cols_to_scale = [c for c in X.select_dtypes(include=np.number).columns
                      if c not in binary_cols + ordinal_cols]

print(f'Бинарные (не масштабируем): {binary_cols}')
print(f'Порядковые (не масштабируем): {ordinal_cols}')
print(f'К масштабированию ({len(num_cols_to_scale)}): {num_cols_to_scale}')

In [ ]:
# Применим все три метода и сравним результат на одном признаке
results = {}
for name in SCALERS:
    X_scaled, _ = scale_features(X, num_cols_to_scale, method=name)
    results[name] = X_scaled

example_col = 'study_hours_per_day'

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].hist(X[example_col], bins=40, color='gray', edgecolor='white')
axes[0].set_title(f'Исходный: {example_col}')

for ax, (name, Xs) in zip(axes[1:], results.items()):
    ax.hist(Xs[example_col], bins=40, color='steelblue', edgecolor='white')
    ax.set_title(f'{name.capitalize()}Scaler')
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)

plt.suptitle(f'Сравнение масштабирования признака «{example_col}»', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# Сводная статистика
summary = pd.DataFrame({
    'Исходный':   X[num_cols_to_scale].agg(['mean', 'std', 'min', 'max']).mean(axis=1),
    'Standard':   results['standard'][num_cols_to_scale].agg(['mean', 'std', 'min', 'max']).mean(axis=1),
    'MinMax':     results['minmax'][num_cols_to_scale].agg(['mean', 'std', 'min', 'max']).mean(axis=1),
    'Robust':     results['robust'][num_cols_to_scale].agg(['mean', 'std', 'min', 'max']).mean(axis=1),
}).round(3)
print('Средние по всем масштабируемым признакам (mean/std/min/max):')
summary

**Выбор метода масштабирования:** поскольку после ЛР №1 выбросы уже ограничены (winsorization), распределения близки к симметричным → используем **StandardScaler** (даёт среднее 0 и дисперсию 1, подходит для линейных моделей и градиентного бустинга).

## 8. Финальная подготовка датасета — единый пайплайн

Соберём все этапы в один `ColumnTransformer` для воспроизводимости и корректного применения fit/transform к train/test раздельно.

In [ ]:
def build_preprocessor(numeric_cols: list[str],
                        nominal_cols: list[str],
                        ordinal_cols_map: dict[str, list]) -> ColumnTransformer:
    """Собирает ColumnTransformer для числовых, номинальных и порядковых признаков."""
    ordinal_cols = list(ordinal_cols_map.keys())
    ordinal_categories = [ordinal_cols_map[c] for c in ordinal_cols]

    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_cols),
            ('nom', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), nominal_cols),
            ('ord', OrdinalEncoder(categories=ordinal_categories,
                                    handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ],
        remainder='passthrough',
        verbose_feature_names_out=False,
    )

In [ ]:
def prepare_dataset(path: str, target: str = 'burnout_score',
                    nrows: int | None = None) -> tuple[pd.DataFrame, pd.Series]:
    """Сквозной пайплайн: загрузка -> очистка -> feature engineering -> X, y."""
    df = load_data(path, nrows=nrows)
    df = basic_clean(df)
    df = add_features(df)

    leaky = ['mental_health_index', 'dropout_risk', 'risk_level']
    y = df[target].copy()
    X = df.drop(columns=[target] + leaky)
    return X, y

# Запускаем сквозной пайплайн «с нуля», чтобы продемонстрировать воспроизводимость
X_raw, y = prepare_dataset('../student_mental_health_burnout_1M.csv', target=TARGET)
print(f'X: {X_raw.shape}, y: {y.shape}')
X_raw.head()

In [ ]:
# Делим train/test ДО fit — корректная практика, чтобы не было утечки статистик из test в train
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape},  Test: {X_test.shape}')

# Определим списки признаков
nominal_cols = ['gender']
ordinal_cols_map = {
    'age_group': ['17-19', '20-22', '23-25', '26-29'],
}
numeric_cols = [c for c in X_train.columns if c not in nominal_cols + list(ordinal_cols_map)]

print(f'\nЧисловые ({len(numeric_cols)}): {numeric_cols}')
print(f'Номинальные ({len(nominal_cols)}): {nominal_cols}')
print(f'Порядковые ({len(ordinal_cols_map)}): {list(ordinal_cols_map)}')

In [ ]:
# Строим и обучаем препроцессор ТОЛЬКО на train
preprocessor = build_preprocessor(numeric_cols, nominal_cols, ordinal_cols_map)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_final = pd.DataFrame(X_train_t, columns=feature_names, index=X_train.index)
X_test_final = pd.DataFrame(X_test_t, columns=feature_names, index=X_test.index)

print(f'X_train_final: {X_train_final.shape}')
print(f'X_test_final:  {X_test_final.shape}')
print(f'\nПризнаки после препроцессинга ({len(feature_names)}):')
print(list(feature_names))

In [ ]:
# Контрольные статистики: numeric признаки должны иметь mean≈0, std≈1 (по train)
print('Средние и стандартные отклонения числовых признаков в train:')
X_train_final[numeric_cols].agg(['mean', 'std']).round(3)

In [ ]:
# Финальный предпросмотр данных для модели
print('=== Итог ===')
print(f'Обучающая выборка:   X = {X_train_final.shape},  y = {y_train.shape}')
print(f'Тестовая выборка:    X = {X_test_final.shape},   y = {y_test.shape}')
print(f'Всего признаков:     {X_train_final.shape[1]}')
print(f'Целевая переменная:  {TARGET} (регрессия)')

X_train_final.head()

In [ ]:
# Сохранение финальных датасетов для дальнейшего моделирования
X_train_final.to_csv('X_train.csv', index=False)
X_test_final.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
print('Финальные файлы сохранены: X_train.csv, X_test.csv, y_train.csv, y_test.csv')

## Итог

В лабораторной работе №2 выполнена подготовка данных к моделированию:

1. **Исправлены замечания по ЛР №1** — устранён баг со взрывным ростом `sleep_study_ratio`, вся логика вынесена в модульные функции.
2. **Проанализированы категориальные признаки** (`gender`, `risk_level`, `academic_year`) — выявлен дисбаланс классов `risk_level` и монотонная связь с целевыми переменными.
3. **Сравнены 3 метода кодирования**: One-Hot, Ordinal, Label. Выбран комбинированный подход — OHE для `gender`, Ordinal для `risk_level` и `age_group`.
4. **Сравнены 3 метода масштабирования**: StandardScaler, MinMaxScaler, RobustScaler. Выбран StandardScaler (данные после winsorization близки к симметричным).
5. **Созданы 6 новых признаков**, из них 2 принципиально новых: `support_to_stress` и `is_sleep_deprived`.
6. **Удалены 3 нерелевантных признака** (`mental_health_index`, `dropout_risk`, `risk_level`) — как источники data leakage для целевой переменной `burnout_score`.
7. **Финальный датасет** собран через `ColumnTransformer` с корректным разделением fit (на train) и transform (на test) — пайплайн готов для обучения любой модели из scikit-learn.